
# Phase 3 — Model Development  
## Claim Outcome Classification Model (03_claim_model.ipynb)

### Business Objective
The goal of this model is to predict the **insurance claim outcome**
(`Paid`, `Pending`, `Rejected`) in order to:

- Identify high rejection risk early
- Reduce revenue leakage
- Improve billing workflow efficiency
- Support financial planning decisions

### Target Variable
`claim_status`

This is a **multi-class classification problem**.

## Step 1 — Import Required Libraries

In [5]:

# Basic data handling
import pandas as pd
import numpy as np

# Machine Learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Model evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Model saving
import joblib
from sklearn.preprocessing import StandardScaler



## Step 2 — Load Modeling Dataset

This dataset was generated in Phase 2 after:
- Data cleaning
- Feature engineering
- SQL-consistent joins

File used: `model_table.csv`


In [6]:

# Load dataset
df = pd.read_csv("model_table.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (25000, 26)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approved_amount,claim_status,payment_days,billing_date,visit_frequency,avg_los_per_patient,provider_rejection_rate,days_since_registration,visit_month,visit_dayofweek
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.00,Rejected,16.0,2025-06-18,2,3.725000,0.148655,65,10,5
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,38178.81,Paid,18.0,2025-10-09,4,32.025000,0.156915,-206,4,6
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,5038.97,Paid,NaN,2025-01-20,4,20.542500,0.149678,9,7,6
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,22813.34,Paid,16.0,2025-12-24,7,28.165714,0.152480,-62,11,2
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,27106.95,Paid,14.0,2025-09-23,5,22.988000,0.149678,0,3,5


## Step 3 — Convert Date Column

We convert `visit_date` to datetime format
to perform a proper time-based train/test split.

In [7]:
df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")

## Step 4 — Define Target Variable

Target: `claim_status`

In [8]:

# Remove rows where claim_status is missing
df = df[df["claim_status"].notna()]

y = df["claim_status"]

print("Claim Status Distribution:")
print(y.value_counts())


Claim Status Distribution:
claim_status
Paid        14940
Pending      6263
Rejected     3797
Name: count, dtype: int64



## Step 5 — Feature Selection (Financial Focus)

We select features relevant to billing and claim processing.

In [9]:

feature_columns = [
    "billed_amount",
    "approved_amount",
    "payment_days",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "provider_rejection_rate",
    "age",
    "chronic_flag"
]

X = df[feature_columns]

# Handle missing values
X = X.fillna(0)



## Step 6 — Time-Based Train/Test Split

As required in the capstone:

- Earliest 80% → Training
- Latest 20% → Testing

This prevents future information leakage.


In [10]:

# Sort dataset by visit_date
df_sorted = df.sort_values("visit_date")

X_sorted = df_sorted[feature_columns].fillna(0)
y_sorted = df_sorted["claim_status"]

# Create split index
split_index = int(len(df_sorted) * 0.8)

# Training data
X_train = X_sorted.iloc[:split_index]
y_train = y_sorted.iloc[:split_index]

# Testing data
X_test = X_sorted.iloc[split_index:]
y_test = y_sorted.iloc[split_index:]

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)


Training Size: (20000, 9)
Testing Size: (5000, 9)


## Step 7 — Baseline Model: Logistic Regression (With Class Weight)

Claim outcomes are often imbalanced.
We use `class_weight='balanced'` to handle imbalance.

In [11]:
# Initialize scaler
scaler = StandardScaler()

# Fit scaler only on training data
scaler.fit(X_train)

# Transform train and test
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Initialize Logistic Regression
baseline_model = LogisticRegression(
    max_iter=3000,        # Increased iterations
    class_weight="balanced",
    solver="lbfgs"
)

# Train model
baseline_model.fit(X_train_scaled, y_train)

# Predict
y_pred_baseline = baseline_model.predict(X_test_scaled)


### Evaluate Baseline Model

In [12]:

print("Confusion Matrix (Baseline):")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nClassification Report (Baseline):")
print(classification_report(y_test, y_pred_baseline))


Confusion Matrix (Baseline):
[[2799    4  194]
 [ 202  982   91]
 [   0    0  728]]

Classification Report (Baseline):
              precision    recall  f1-score   support

        Paid       0.93      0.93      0.93      2997
     Pending       1.00      0.77      0.87      1275
    Rejected       0.72      1.00      0.84       728

    accuracy                           0.90      5000
   macro avg       0.88      0.90      0.88      5000
weighted avg       0.92      0.90      0.90      5000



## Step 8 — Advanced Model: Random Forest

Random Forest handles nonlinear relationships and imbalance better.

In [13]:

rf_model_tuned = RandomForestClassifier(
    n_estimators=500,       # More trees
    max_depth=20,           # Allow deeper trees
    min_samples_split=4,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1               # Use all CPU cores
)

rf_model_tuned.fit(X_train, y_train)

y_pred_rf_tuned = rf_model_tuned.predict(X_test)



### Evaluate Random Forest Model

In [14]:

print("Confusion Matrix (Tuned RF):")
print(confusion_matrix(y_test, y_pred_rf_tuned))

print("\nClassification Report (Tuned RF):")
print(classification_report(y_test, y_pred_rf_tuned))



Confusion Matrix (Tuned RF):
[[2838   29  130]
 [ 111 1108   56]
 [  18    1  709]]

Classification Report (Tuned RF):
              precision    recall  f1-score   support

        Paid       0.96      0.95      0.95      2997
     Pending       0.97      0.87      0.92      1275
    Rejected       0.79      0.97      0.87       728

    accuracy                           0.93      5000
   macro avg       0.91      0.93      0.91      5000
weighted avg       0.94      0.93      0.93      5000



## Step 9 — Save Best Model

We save the Random Forest model for deployment.

In [16]:

joblib.dump(rf_model_tuned, "claim_model.pkl")
print("Claim model saved successfully as claim_model.pkl")

Claim model saved successfully as claim_model.pkl


In [24]:
# -----------------------------------
# Create Claim Feature Schema JSON
# -----------------------------------

import json

# Get feature names from training data
claim_features = list(X_train.columns)

claim_schema = {
    "model_name": "Claim Outcome Classification Model",
    "model_type": "RandomForestClassifier",
    "target_variable": "claim_status",
    "features": [
        {"name": col, "type": "numeric"} for col in claim_features
    ],
    "class_labels": list(y_train.unique()),
    "preprocessing": {
        "missing_value_strategy": "Numerical features filled with 0",
        "scaling_required_for_logistic_regression": True,
        "scaling_required_for_random_forest": False
    },
    "train_test_split": {
        "method": "Time-based split",
        "training_percentage": 80,
        "testing_percentage": 20
    }
}

with open("03_claim_feature_schema.json", "w") as f:
    json.dump(claim_schema, f, indent=4)

print("Claim feature schema saved successfully.")

Claim feature schema saved successfully.
